# Perishable Product Demand Forecasting

An end-to-end workflow for estimating underlying daily demand when retail products experience stockouts, then forecasting demand for the next seven days.

This notebook documents and orchestrates the project's existing Python pipeline. The implementation is maintained in the `src/` modules; the notebook provides a reproducible, narrative walkthrough of the data, methodology, experiments, and evaluation.

> **Scope:** Daily demand forecasting only. The forecasting approaches evaluated are a Naive baseline and Ridge Regression. The earlier demand-recovery stage is a preprocessing step, not the final seven-day forecasting model.


## 1. Project objective

Observed sales can understate customer demand when a product is unavailable. The workflow therefore has two stages:

1. **Historical demand recovery:** estimate daily demand for historical store-product observations affected by stockouts.
2. **Seven-day forecasting:** use recovered historical demand and lag-based features to predict daily demand for each store-product combination over the next seven days.

The dataset does not provide independently verified lost-demand ground truth. Recovery performance is evaluated by artificially masking portions of fully in-stock observations and comparing reconstructed demand with the original observed demand. This is a methodological proxy, not proof of accuracy during real stockouts.


## 2. Environment and project setup

Run this notebook from the repository root or from the `notebooks/` directory; the first cell locates the project root automatically. The project uses `uv` to manage its Python environment.

The following cell checks the working directory and expected project files.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

required_paths = [
    PROJECT_ROOT / "pyproject.toml",
    PROJECT_ROOT / "src" / "perishable_demand_forecasting",
    PROJECT_ROOT / "data",
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run this notebook from the project root. Missing: " + ", ".join(missing)
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")
print("Project structure checks passed.")


### Dependencies

Install the project dependencies from the repository root before running the notebook:

```bash
uv sync
uv add --dev ipykernel jupyter
uv run jupyter notebook
```

If Jupyter is already installed in the project environment, the second command is unnecessary.


## 3. Dataset inspection, data cleaning and exploratory analysis

The project uses the FreshRetailNet-50K dataset. The raw training data covers 2024-03-28 through 2024-06-25; the evaluation data covers 2024-06-26 through 2024-07-02.

The next cells run the existing inspection and exploratory-analysis modules. They validate the source schema, date coverage, missing values, duplicate keys, hourly arrays, sales reconciliation, and stockout patterns.


In [ ]:
def run_module(module_name: str):
    """Run an existing project module with the repository's uv environment."""
    command = ["uv", "run", "python", "-m", module_name]
    print("$", " ".join(command))

    # Force UTF-8 output so Unicode characters printed by the scripts
    # are decoded correctly on Windows and other operating systems.
    env = os.environ.copy()
    env["PYTHONIOENCODING"] = "utf-8"

    result = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"{module_name} failed with exit code {result.returncode}")
    return result.stdout


### What this script does

**Inputs**

Raw inputs: `data/raw/freshretailnet/data/train.parquet` and `eval.parquet`.

**Processing**

Checks schema, row counts, date/entity coverage, nulls, duplicates, hourly-array lengths, sales reconciliation, stockout reconciliation, and value ranges.

**Outputs**

No model or dataset transformation is produced; this is a quality gate before analysis.

**Why this step matters**

Confirms the raw data is structurally consistent before downstream calculations depend on it.


In [ ]:
run_module("perishable_demand_forecasting.analysis.inspect_raw_dataset")

### What this script does

**Inputs**

Raw hourly sales and stock-status arrays from the FreshRetailNet data.

**Processing**

Expands the hourly arrays, restricts analysis to the 06:00–21:00 operating window, and summarizes sales and stockout frequency by hour.

**Outputs**

Produces exploratory diagnostic output; it does not train a model or alter the raw data.

**Why this step matters**

Shows when demand is observed and whether stockouts are concentrated at particular hours.


In [ ]:
run_module("perishable_demand_forecasting.analysis.analyze_hourly_demand_patterns")

### What this script does

**Inputs**

Daily sales and stockout information from the raw dataset.

**Processing**

Groups comparable store-product observations and compares sales on stockout-affected versus fully in-stock days.

**Outputs**

Produces observational comparison statistics; it does not infer causality.

**Why this step matters**

Helps determine whether stockout-affected observations systematically differ from in-stock observations while acknowledging reverse causality.


In [ ]:
run_module("perishable_demand_forecasting.analysis.analyze_matched_stockouts")

### What this script does

**Inputs**

Daily stockout-hour counts and hourly stock-status/sales information.

**Processing**

Summarizes stockout severity, observed sales by severity, and hourly stockout patterns.

**Outputs**

Produces stockout diagnostics used to understand the scale and severity of availability constraints.

**Why this step matters**

Provides evidence for designing the artificial masking experiment used to evaluate demand recovery.


In [ ]:
run_module("perishable_demand_forecasting.analysis.analyze_stockouts")

### Key data observations

- The source contains approximately 4.85 million daily store-product observations across 97 dates.
- Stockout-affected observations account for approximately 44.27% of training rows and 40.98% of evaluation rows.
- Fully out-of-stock observations account for approximately 4.04% of training rows and 3.58% of evaluation rows.
- Hourly sales and daily `sale_amount` reconcile within floating-point precision.
- Some sales are recorded in hours labelled out of stock, so those observations are not automatically treated as zero sales.
- Stockout patterns are observational and can be affected by reverse causality: high demand may itself contribute to stockouts.


### Data preparation and cleaning summary

The pipeline performs the following cleaning and validation steps before modelling:

- Parse `dt` as a date and enforce a 16-value hourly array for the 06:00–21:00 operating window.
- Reconcile daily `sale_amount` with the sum of `hours_sale` within floating-point tolerance.
- Check for missing values, duplicate store-product-date keys, and out-of-range stock-status values; downstream builders reject invalid hourly arrays.
- Build recovery examples from fully in-stock observations and borrow stockout masks from separate observations.
- Clip recovered daily demand to zero before lag construction, because negative demand is not physically meaningful.
- Use chronological train/validation/test splits, with a seven-day embargo between forecasting partitions to prevent temporal leakage.


## 4. Historical demand recovery

The recovery dataset is created using fully in-stock observations as reconstruction targets. Stockout masks are borrowed from other observations and applied to the hourly sales profile. The recovery model learns to estimate the original daily demand from the masked observations and associated features.

The data is split chronologically into training, validation, and test periods. The test period is held out from model selection.


### What this script does

**Inputs**

Raw observations that are fully in stock and therefore have complete observed daily sales.

**Processing**

Uses those observations as reconstruction targets and applies stockout masks borrowed from other observations to their hourly sales. It creates chronological train, validation, and test recovery datasets.

**Outputs**

Saves masked recovery examples under `data/processed/freshretailnet/recovery/`.

**Why this step matters**

Creates a controlled reconstruction problem where the original daily demand is known. This is a methodological proxy, not independently verified lost-demand ground truth.


In [ ]:
run_module("perishable_demand_forecasting.recovery.build_recovery_dataset")

### What this script does

**Inputs**

The artificially masked recovery validation dataset.

**Processing**

Calculates simple reference predictions, including the observed masked sales value and a mean-based prediction, and measures their errors against the original target demand.

**Outputs**

Saves baseline recovery metrics. No learned model is trained.

**Why this step matters**

Establishes a minimum reference point that the learned recovery model must improve upon.


In [ ]:
run_module("perishable_demand_forecasting.recovery.evaluate_recovery_baseline")

### What this script does

**Inputs**

Recovery train and validation datasets.

**Processing**

Adds calendar variables and prepares categorical, business, weather, simulated stockout, and hourly-sales information for modelling.

**Outputs**

Saves model-ready feature tables for train and validation.

**Why this step matters**

Creates a consistent feature base so later recovery experiments can be compared fairly.


In [ ]:
run_module("perishable_demand_forecasting.recovery.build_recovery_features")

### What this script does

**Inputs**

The prepared recovery feature tables.

**Processing**

Reports dimensions, target name, candidate features, data types, and missing values.

**Outputs**

Produces a feature-quality inspection report.

**Why this step matters**

Verifies that the expected modelling inputs exist before running feature experiments.


In [ ]:
run_module("perishable_demand_forecasting.recovery.inspect_recovery_features")

### What this script does

**Inputs**

The prepared recovery feature tables.

**Processing**

Builds predefined feature groups A–H by progressively adding product, business, weather, hourly, and management information to a common core.

**Outputs**

Writes one train/validation dataset per experiment under `features/experiments/`.

**Why this step matters**

Allows feature-group performance to be compared under the same recovery modelling setup.


In [ ]:
run_module("perishable_demand_forecasting.recovery.select_recovery_features")

### What this script does

**Inputs**

The A–H recovery experiment datasets.

**Processing**

For each experiment, encodes categorical identifiers, scales numerical features, trains Ridge Regression on the training partition, and evaluates it on validation data.

**Outputs**

Saves validation metrics and Ridge model artifacts; experiment G is selected based on validation performance.

**Why this step matters**

Identifies the feature configuration that provides the strongest recovery performance without using the held-out test set.


In [ ]:
run_module("perishable_demand_forecasting.recovery.train_recovery_experiments")

### What this script does

**Inputs**

The selected recovery model and its validation predictions.

**Processing**

Groups reconstruction errors by the number of artificially masked stockout hours and compares Ridge with the observed-sales baseline.

**Outputs**

Produces severity-level MAE/RMSE/bias diagnostics.

**Why this step matters**

Shows whether recovery performance changes as the amount of missing demand information increases.


In [ ]:
run_module("perishable_demand_forecasting.recovery.evaluate_recovery_by_stockout_severity")

### What this script does

**Inputs**

Recovery experiment G plus its train/validation data.

**Processing**

Trains Ridge models using several predefined `alpha` values and compares validation MAE, RMSE, bias, and negative-prediction rate.

**Outputs**

Saves regularization metrics and candidate model artifacts.

**Why this step matters**

Tests whether regularization materially improves the selected recovery model before freezing the configuration.


In [ ]:
run_module("perishable_demand_forecasting.recovery.tune_recovery_ridge")

### What this script does

**Inputs**

The recovery validation predictions from Ridge alpha 1 and alpha 10.

**Processing**

Compares the two regularization settings with the observed-sales baseline separately across stockout-severity groups.

**Outputs**

Produces severity-level comparisons for the candidate recovery configurations.

**Why this step matters**

Checks whether a regularization choice behaves differently for lightly versus heavily masked observations.


In [ ]:
run_module("perishable_demand_forecasting.recovery.evaluate_tuned_ridge_by_severity")

### What this script does

**Inputs**

Validation predictions from the observed-sales baseline, Ridge alpha 1, and Ridge alpha 10.

**Processing**

Applies a fixed severity-based routing rule: observed sales for 1–5 stockout hours, Ridge alpha 10 for 6–9 hours, and Ridge alpha 1 for 10–16 hours.

**Outputs**

Reports overall and severity-level validation performance for the hybrid and its component models.

**Why this step matters**

Tests whether combining methods according to known historical stockout severity improves reconstruction accuracy.


In [ ]:
run_module("perishable_demand_forecasting.recovery.evaluate_hybrid_strategies")

### What this script does

**Inputs**

The frozen hybrid strategy plus several alternative fixed routing strategies.

**Processing**

Compares a small predefined set of routing combinations without searching endlessly for thresholds.

**Outputs**

Produces validation metrics for each routing strategy and identifies the frozen hybrid.

**Why this step matters**

Checks whether the chosen recovery rule is reasonably stable rather than depending on one arbitrary Ridge assignment.


In [ ]:
run_module("perishable_demand_forecasting.recovery.test_hybrid_robustness")

### What this script does

**Inputs**

The untouched artificial-mask recovery test set and the frozen recovery strategy.

**Processing**

Applies the frozen hybrid without changing its rules, then measures reconstruction error overall and by stockout severity.

**Outputs**

Produces the final recovery test metrics.

**Why this step matters**

Provides the held-out evaluation of the recovery stage while preserving the limitation that the target comes from artificial masking.


In [ ]:
run_module("perishable_demand_forecasting.recovery.evaluate_hybrid_on_test")

### Recovery model selection and test results

The selected recovery approach is a frozen hybrid strategy:

- 0–5 stockout hours: use observed daily sales.
- 6–9 stockout hours: use Ridge Regression with alpha 10.
- 10–16 stockout hours: use Ridge Regression with alpha 1.

The untouched artificial-mask test evaluation produced:

| Method | MAE | RMSE |
|---|---:|---:|
| Observed-sales baseline | 0.418767 | 0.977165 |
| Frozen hybrid recovery | **0.259736** | **0.587402** |

The hybrid reduced MAE by approximately 38.0% relative to the observed-sales baseline on this artificial-mask test. This result should be interpreted within the reconstruction experiment and not as independently verified real-stockout accuracy.

Negative recovery predictions are clipped to zero in the historical preprocessing pipeline. The underlying saved Ridge models are not modified.


## 5. Prepare the seven-day forecasting dataset

The forecasting dataset is built from the recovered daily demand history. It includes:

- Demand lags 1–14.
- Seven-day and fourteen-day rolling mean demand.
- Lagged stockout information.
- Forecast horizon (days 1–7).
- Future-known calendar metadata such as weekday, month, and holiday flag.

The data is split chronologically into training, validation, and test partitions, with a seven-day embargo between partitions to reduce temporal leakage.


### What this script does

**Inputs**

Raw historical train/eval observations plus the frozen recovery Ridge models.

**Processing**

Recovers historical daily demand using the frozen hybrid, builds demand lags 1–14, rolling means, lagged stockout indicators, and seven forecast horizons, then performs chronological splitting with a seven-day embargo.

**Outputs**

Saves `recovered_daily_demand.parquet`, `train.parquet`, `validation.parquet`, and `test.parquet` under `forecasting/`.

**Why this step matters**

Transforms the recovered historical series into the supervised dataset required by the final seven-day forecasting models.


In [ ]:
run_module("perishable_demand_forecasting.data.build_daily_forecast_dataset")

### Forecasting dataset summary

The successful dataset preparation produced:

| Partition | Forecast origins | Rows |
|---|---:|---:|
| Training | 37 | 12,950,000 |
| Validation | 13 | 4,550,000 |
| Test | 12 | 4,200,000 |

The forecast target is recovered daily demand. The recovery stage is historical preprocessing; it is not the seven-day forecasting model.


### Target variable analysis

The forecasting target is recovered daily demand. The summary checks its range, central tendency, tail and zero share across the chronological splits; the histogram uses the training split.


In [ ]:
import matplotlib.pyplot as plt
import polars as pl

from perishable_demand_forecasting.config import FORECASTING_DIR

summary_rows = []
for split in ["train", "validation", "test"]:
    target = (
        pl.scan_parquet(FORECASTING_DIR / f"{split}.parquet")
        .select(pl.col("target_daily_demand"))
        .collect()
        .get_column("target_daily_demand")
    )
    summary_rows.append(
        {
            "split": split,
            "rows": target.len(),
            "min": target.min(),
            "median": target.median(),
            "mean": target.mean(),
            "p99": target.quantile(0.99),
            "max": target.max(),
            "zero_pct": (target == 0).mean() * 100,
        }
    )

print(pl.DataFrame(summary_rows).to_pandas().to_string(index=False))

train_target = (
    pl.scan_parquet(FORECASTING_DIR / "train.parquet")
    .select(pl.col("target_daily_demand"))
    .collect()
    .get_column("target_daily_demand")
    .to_numpy()
)

plt.figure(figsize=(9, 4))
plt.hist(train_target, bins=80, color="#2471a3")
plt.xlabel("Recovered daily demand")
plt.ylabel("Observations")
plt.title("Forecasting target distribution (training split)")
plt.tight_layout()
plt.show()


## 6. Naive forecasting baseline

The Naive baseline predicts each horizon using the most recent available recovered daily demand (`demand_lag_1`). It is rule-based and therefore has no model-training step.

The evaluation reports MAE, RMSE, and bias overall and separately for each forecast horizon.


### What this script does

**Inputs**

The forecasting validation dataset.

**Processing**

Uses `demand_lag_1` as the prediction for every forecast horizon, with no training step. It calculates MAE, RMSE, and bias overall and for days 1–7.

**Outputs**

Saves Naive validation metrics.

**Why this step matters**

Provides the simple baseline that Ridge must beat to demonstrate useful predictive value.


In [ ]:
run_module("perishable_demand_forecasting.forecasting.evaluate_naive_forecast")

### Naive validation results

| Metric | Validation |
|---|---:|
| MAE | 0.477620 |
| RMSE | 0.803075 |
| Bias | +0.020007 |



## 7. Linear forecasting model (Ridge)

Ridge Regression is the linear baseline. It is trained using the training partition. Feature scaling is fitted on training data only and then applied to validation data. The selected feature set includes demand lags, rolling statistics, lagged stockout information, forecast horizon, and future-known calendar fields.

The initial configuration uses `alpha=1.0`. The model artifact stores the fitted scaler, estimator, feature list, target name, and alpha.


### What this script does

**Inputs**

Forecasting train and validation datasets.

**Processing**

Selects the 22 forecasting features, fits `StandardScaler` only on training data, trains Ridge Regression with alpha 1.0, and evaluates validation predictions.

**Outputs**

Saves the Ridge model, scaler, feature metadata, and validation metrics.

**Why this step matters**

Creates the main machine-learning forecasting model while preventing validation information from influencing feature scaling.


In [ ]:
run_module("perishable_demand_forecasting.forecasting.train_ridge_forecast")

### Ridge regularization experiment

Several alpha values were evaluated on the validation partition. The displayed metrics were effectively indistinguishable at the reported precision:

| Alpha | MAE | RMSE |
|---:|---:|---:|
| 0.01 | 0.374763 | 0.604713 |
| 0.1 | 0.374763 | 0.604713 |
| 1 | 0.374763 | 0.604713 |
| 10 | 0.374763 | 0.604713 |
| 100 | 0.374763 | 0.604713 |

No meaningful practical improvement was demonstrated by increasing alpha. The project therefore retains the original `alpha=1.0` model for the final test evaluation rather than selecting a value based on negligible metric differences.


### What this script does

**Inputs**

The same forecasting train/validation partitions used for the initial Ridge model.

**Processing**

Evaluates Ridge across predefined alpha values using the same features and training-only scaler.

**Outputs**

Saves alpha comparison metrics and a tuned candidate artifact.

**Why this step matters**

Checks whether changing regularization produces a meaningful validation improvement; the differences were negligible, so alpha 1 remains the selected final configuration.


In [ ]:
run_module("perishable_demand_forecasting.forecasting.tune_ridge_forecast")

### Feature importance analysis

The final model is a Random Forest, so its built-in `feature_importances_` (impurity-based) are used. If a linear artifact is loaded instead, the absolute standardized Ridge coefficients are shown. The final model is retrained on the train+validation partitions by `train.py`; the notebook uses that artifact.


In [ ]:
import joblib
import pickle
import matplotlib.pyplot as plt
import polars as pl

from perishable_demand_forecasting.config import FORECASTING_MODELS_DIR

model_candidates = [
    FORECASTING_MODELS_DIR / "final_demand_forecast.pkl",
    FORECASTING_MODELS_DIR / "tree_forecast_tuned.pkl",
    FORECASTING_MODELS_DIR / "ridge_forecast.joblib",
]
model_path = next((path for path in model_candidates if path.exists()), None)
if model_path is None:
    raise FileNotFoundError(
        "No trained model found. Run `uv run python train.py` first."
    )

if model_path.suffix == ".pkl":
    with model_path.open("rb") as file:
        bundle = pickle.load(file)
else:
    bundle = joblib.load(model_path)

print(f"Using model: {model_path.name} ({bundle.get('family', 'ridge_regression')})")

model = bundle["model"]
if hasattr(model, "feature_importances_"):
    values = model.feature_importances_
    value_label = "importance"
    value_title = "Random Forest feature importance"
else:
    values = model.coef_
    value_label = "coefficient"
    value_title = "Standardized Ridge coefficient"

importance = (
    pl.DataFrame({"feature": bundle["features"], value_label: values})
    .with_columns(
        pl.col(value_label).abs().alias("absolute_value"),
        pl.when(pl.col("feature").str.starts_with("demand_lag_"))
        .then(pl.lit("Demand lags"))
        .when(pl.col("feature").str.starts_with("demand_rolling"))
        .then(pl.lit("Rolling demand"))
        .when(pl.col("feature").str.starts_with("stockout_hours"))
        .then(pl.lit("Stockout history"))
        .when(pl.col("feature") == "forecast_horizon")
        .then(pl.lit("Forecast horizon"))
        .otherwise(pl.lit("Calendar"))
        .alias("feature_group"),
    )
    .sort("absolute_value", descending=True)
)

print()
print("Feature importance (sorted by absolute value):")
print(importance.to_pandas().to_string(index=False))

group_importance = (
    importance.group_by("feature_group")
    .agg(pl.col("absolute_value").sum().alias("total_absolute_value"))
    .with_columns(
        (pl.col("total_absolute_value") / pl.col("total_absolute_value").sum() * 100)
        .round(2)
        .alias("share_pct")
    )
    .sort("total_absolute_value", descending=True)
)

print()
print("Aggregated importance by feature group:")
print(group_importance.to_pandas().to_string(index=False))

top = importance.head(12).sort(value_label)
colors = (
    ["#2471a3"] * top.height
    if value_label == "importance"
    else ["#c0392b" if value < 0 else "#2471a3" for value in top[value_label].to_list()]
)

plt.figure(figsize=(9, 5))
plt.barh(top["feature"].to_list(), top[value_label].to_list(), color=colors)
if value_label == "coefficient":
    plt.axvline(0, color="grey", linewidth=0.8)
plt.xlabel(value_title)
plt.title("Top forecasting feature importances")
plt.tight_layout()
plt.show()


### Feature importance interpretation

The two rolling demand means dominate the forest, followed by the shortest demand lags; calendar fields and stockout history add smaller adjustments. This matches the strong autocorrelation of daily demand and supports the lag-based feature design. The near-zero importance of `forecast_horizon` means the model forecasts the historical level rather than adapting per horizon, which is a limitation to report.


## 8. Tree-based forecasting models

This comparison covers decision trees, random forests, and gradient boosting. Three candidates are tuned on deterministic samples of the training partition and scored on the full validation partition with the same MAE/RMSE/bias protocol used for Ridge:

- **Decision Tree Regressor** — tuned `max_depth` and `min_samples_leaf`.
- **Random Forest Regressor** — tuned `max_depth` and `min_samples_leaf` (60 trees).
- **XGBoost Regressor** — tuned `learning_rate` and `max_depth` with early stopping.

Sampling keeps the experiment reproducible and runnable on a laptop; every candidate is still evaluated on all 4.55M validation rows.


### What this script does

**Inputs**

Forecasting train and validation datasets.

**Processing**

Draws deterministic training samples, tunes decision-tree depth/leaf size, random-forest depth/leaf size, and XGBoost learning rate/depth with early stopping, then scores every candidate on the full validation partition.

**Outputs**

Saves candidate validation metrics and the best tree model artifact.

**Why this step matters**

Implements the linear-versus-tree model comparison required for model selection and checks whether non-linear splits beat the linear Ridge baseline.


In [ ]:
run_module("perishable_demand_forecasting.forecasting.tune_tree_forecast")

### Tree model validation results

| Model | Tuned settings | Validation MAE | RMSE | Bias |
|---|---|---:|---:|---:|
| Naive (lag 1) | — | 0.477620 | 0.803075 | 0.020007 |
| Ridge Regression | alpha=1.0 | 0.374763 | 0.604713 | 0.001554 |
| Decision Tree | depth=12, leaf=100 | 0.368541 | 0.602604 | 0.032915 |
| XGBoost | lr=0.05, depth=10 | 0.368517 | 0.640420 | 0.022574 |
| Random Forest | depth=14, leaf=20, 60 trees | **0.361513** | **0.577953** | 0.031059 |

The Random Forest has the lowest validation MAE and is selected as the final model. `train.py` retrains it on the full train + validation partitions before serving.


In [ ]:
import matplotlib.pyplot as plt
import polars as pl

from perishable_demand_forecasting.config import FORECASTING_DIR

tree_metrics = pl.read_csv(
    FORECASTING_DIR / "tree_validation_metrics.csv"
).sort("mae")
ridge_metrics = pl.read_csv(FORECASTING_DIR / "ridge_validation_metrics.csv")
naive_metrics = pl.read_csv(FORECASTING_DIR / "naive_validation_metrics.csv")

comparison = pl.concat(
    [
        pl.DataFrame(
            {
                "model": ["naive"],
                "mae": [
                    naive_metrics.filter(
                        pl.col("evaluation_scope") == "overall"
                    )["mae"][0]
                ],
            }
        ),
        pl.DataFrame(
            {
                "model": ["ridge"],
                "mae": [
                    ridge_metrics.filter(
                        pl.col("evaluation_scope") == "overall"
                    )["mae"][0]
                ],
            }
        ),
        tree_metrics.group_by("model").agg(pl.col("mae").min()).sort("mae"),
    ]
)

plt.figure(figsize=(8, 4))
plt.bar(
    comparison["model"].to_list(),
    comparison["mae"].to_list(),
    color=["#7f8c8d", "#2471a3", "#c0392b", "#e67e22", "#27ae60"][
        : comparison.height
    ],
)
plt.ylabel("Validation MAE")
plt.title("Linear vs tree-based validation performance")
plt.tight_layout()
plt.show()


## 9. Final forecasting evaluation

The final evaluation uses the untouched chronological test partition. The Naive baseline, Ridge Regression, and the selected Random Forest are evaluated against the same target rows. Negative predictions are reported rather than silently clipped during evaluation.


### What this script does

**Inputs**

The untouched forecasting test partition and the trained Ridge and Random Forest artifacts.

**Processing**

Generates seven-day predictions for both models and compares them with the Naive baseline on exactly the same test rows.

**Outputs**

Saves overall and horizon-level test metrics, including MAE, RMSE, bias, and negative-prediction rate.

**Why this step matters**

Provides the final unbiased comparison between the forecasting approaches on data not used for model selection.


In [ ]:
run_module("perishable_demand_forecasting.forecasting.evaluate_ridge_forecast_test")

### Tree model test evaluation

The tuned Random Forest (and, when present, the final full-data `train.py` artifact) is scored on the test partition with the same protocol.


In [ ]:
run_module("perishable_demand_forecasting.forecasting.evaluate_tree_forecast_test")

### Final test results

| Model | MAE | RMSE | Bias | Negative predictions |
|---|---:|---:|---:|---:|
| Naive baseline | 0.512812 | 0.814507 | -0.037006 | 0.00% |
| Ridge Regression (alpha=1) | 0.407412 | 0.648432 | -0.114638 | 3.54% |
| Random Forest (final) | **0.399386** | 0.651719 | -0.107226 | 0.00% |

The selected Random Forest reduced test MAE by approximately 22.12% versus the Naive baseline and by approximately 1.97% versus Ridge. Ridge keeps a slightly lower RMSE (0.6484 vs 0.6517). Both models have a negative bias, and Random Forest avoids negative predictions entirely; these limitations are reported transparently.


## 10. Reproducibility and generated artifacts

The pipeline writes intermediate datasets, metrics, and model artifacts to the following locations:

- `data/processed/freshretailnet/recovery/`
- `data/processed/freshretailnet/features/`
- `data/processed/freshretailnet/forecasting/`
- `models/recovery/`
- `models/forecasting/`

The notebook calls the existing modules rather than duplicating their implementations. Run the cells from top to bottom in a clean project environment. Some stages process millions of rows and can require substantial memory and execution time.

For deployment, `train.py` fits the final Random Forest model on the train+validation partitions and saves it to `models/forecasting/final_demand_forecast.pkl`. `predict.py` loads that artifact and serves demand forecasts through a Flask API (`uv run python predict.py`).
